In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("All tools loaded. Ready to build an auto-router.")


In [ ]:
np.random.seed(42)

network_templates = [
    "The network is very weak in {place}, calls keep dropping",
    "No internet connection for two days in {place}",
    "My data speed is extremely slow today",
    "I am getting no signal at all near {place}",
    "The 5G network in {place} keeps disconnecting",
    "Internet is down in my area since this morning",
    "Poor coverage inside my building in {place}",
    "Calls keep dropping every few minutes in {place}",
    "My connection is unstable since yesterday",
    "There is no network signal in {place}",
]
billing_templates = [
    "My bill amount is higher than expected this month",
    "I was charged twice for the same plan",
    "Please explain the extra charges on my invoice",
    "I want a refund for an incorrect charge",
    "The National Day offer was not applied to my bill",
    "My last payment does not appear on my account",
    "There is a duplicate charge on my statement",
    "Why is my bill different from last month",
    "I was overcharged and need this corrected",
    "Can you explain the charges on my last invoice",
]
sim_templates = [
    "My new SIM card is not activating",
    "I cannot activate the eSIM I purchased",
    "SIM activation failed twice today",
    "How do I activate international roaming for {place}",
    "My SIM shows no service after activation",
    "I need help activating my replacement SIM",
    "The eSIM QR code is not working for activation",
    "My SIM is not activating please help",
    "I bought a new eSIM and it will not activate",
    "Activation of my SIM keeps failing",
]
plan_templates = [
    "I want to upgrade to a bigger data plan",
    "Please help me switch to postpaid",
    "I would like to downgrade my current plan",
    "Can I change my monthly plan to unlimited data",
    "I want to switch to a family plan",
    "How do I change my subscription plan",
    "I need to move to a cheaper plan",
    "I would like to switch to postpaid",
    "What plans do you have for more data",
    "I want to upgrade my current plan",
]

places = ["Muscat", "Salalah", "Sohar", "Nizwa", "Sur"]


def fill(t):
    return t.replace("{place}", np.random.choice(places)) if "{place}" in t else t


rows = []
for _ in range(70):
    rows.append((fill(np.random.choice(network_templates)), "Network"))
for _ in range(64):
    rows.append((fill(np.random.choice(billing_templates)), "Billing"))
for _ in range(53):
    rows.append((fill(np.random.choice(sim_templates)), "SIM_Activation"))
for _ in range(58):
    rows.append((fill(np.random.choice(plan_templates)), "Plan_Change"))

df = pd.DataFrame(rows, columns=["ticket_text", "team"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, "ticket_id", [f"TKT{70000 + i}" for i in range(len(df))])

df.to_csv("omantel_support_tickets.csv", index=False)
print(f"Created {len(df)} unique support tickets across 4 teams.")
print(df["team"].value_counts())


In [ ]:
df = pd.read_csv("omantel_support_tickets.csv")
print(f"Loaded {len(df)} tickets from the CSV file.")


In [ ]:
df.head(10)


In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["ticket_text"])
y = df["team"]
print(f"Each ticket is now {X.shape[1]} number-columns the machine can read.")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Training tickets: {X_train.shape[0]}   Exam tickets: {X_test.shape[0]}")


In [ ]:
router = LogisticRegression(max_iter=1000)
router.fit(X_train, y_train)
print("Router trained.")


In [ ]:
preds = router.predict(X_test)
acc = accuracy_score(y_test, preds) * 100
print(f"Routing accuracy: {acc:.1f}%")
print(classification_report(y_test, preds))


In [ ]:
labels = sorted(df["team"].unique())
cm = confusion_matrix(y_test, preds, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)


In [ ]:
hard = [
    "My new plan shows the wrong price on my bill",
    "I changed my plan but now the internet does not work",
    "The offer charged me but my SIM is not active",
    "Slow internet and my bill went up too",
]
probs = router.predict_proba(vectorizer.transform(hard))
classes = router.classes_
for t, pr in zip(hard, probs):
    top = sorted(zip(classes, pr), key=lambda x: -x[1])[:2]
    print(f"{t}")
    print(
        f"   -> {top[0][0]} ({top[0][1] * 100:.0f}%) | 2nd guess: {top[1][0]} ({top[1][1] * 100:.0f}%)\n"
    )


In [ ]:
new_tickets = [
    "My bill is too high this month please check",
    "There is no network signal in Nizwa",
    "I cannot activate my new eSIM",
    "I want to upgrade to a bigger data plan",
]
routes = router.predict(vectorizer.transform(new_tickets))
print("LIVE ROUTING:")
for t, r in zip(new_tickets, routes):
    print(f"   -> [{r:15}]  {t}")
